In [ ]:
# Imports and Earth Engine Initialization
import ee
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import savgol_filter

# ee.Authenticate()
ee.Initialize(project='qsair-463811')


In [ ]:
# Define the field boundary (Field near Kampala)
field_boundary = ee.Geometry.Polygon([
    [
        [32.570, 0.360], # Southwest corner
        [32.570, 0.380], # Northwest corner
        [32.590, 0.380], # Northeast corner
        [32.590, 0.360], # Southeast corner
        [32.570, 0.360]  # Close polygon back to SW corner
    ]
])


In [ ]:
# Define date range and bands for NDVI calculation
start_date_filter = '2018-01-01'
end_date_filter = '2023-12-31'
input_bands = ['B8', 'B4']  # NIR and Red bands for NDVI
output_band = 'NDVI'

In [ ]:
# Function to add NDVI band to Sentinel-2 images
def addNDVI(image):
    ndvi = image.normalizedDifference(input_bands).rename(output_band)
    return image.addBands(ndvi)


In [ ]:
# Filter Sentinel-2 ImageCollection by date, location, and cloud cover
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterDate(start_date_filter, end_date_filter)
                 .filterBounds(field_boundary)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))


In [ ]:
# Map NDVI calculation over the filtered ImageCollection
s2_collection_with_ndvi = s2_collection.map(addNDVI)

In [ ]:
# Extract NDVI time series for the field boundary
ndvi_data = s2_collection_with_ndvi.select(output_band).getRegion(field_boundary, 30).getInfo()

In [ ]:
# Convert NDVI data to a Pandas DataFrame and prepare time index
ndvi_data_df = pd.DataFrame(ndvi_data[1:], columns=ndvi_data[0])
ndvi_data_df['time'] = pd.to_datetime(ndvi_data_df['time'], unit='ms')
ndvi_data_df.set_index('time', inplace=True)
ndvi_data_df = ndvi_data_df[[output_band]]


In [ ]:
# Add 'year' and 'sort_day' columns for time series analysis and sorting
ndvi_data_df['year'] = ndvi_data_df.index.year
ndvi_data_df['sort_day'] = ndvi_data_df.index.dayofyear + (ndvi_data_df.index.hour + ndvi_data_df.index.minute/60) / 24
ndvi_data_df.sort_values(by='sort_day', inplace=True)


In [ ]:
# Smooth NDVI time series using Savitzky-Golay filter
window_length = 31  # Must be odd and > polyorder
polyorder = 2

if len(ndvi_data_df) >= window_length:
    ndvi_data_df['NDVI_smooth'] = savgol_filter(ndvi_data_df[output_band], window_length, polyorder)
else:
    print(f"Warning: Not enough data points ({len(ndvi_data_df)}) for smoothing. Skipping smoothing.")
    ndvi_data_df['NDVI_smooth'] = ndvi_data_df[output_band]


In [ ]:
# Plot raw and smoothed NDVI time series
plt.figure(figsize=(12, 6))
plt.plot(ndvi_data_df.index, ndvi_data_df[output_band], label=f'Raw {output_band} values', alpha=0.5)

if 'NDVI_smooth' in ndvi_data_df.columns:
    plt.plot(ndvi_data_df.index, ndvi_data_df['NDVI_smooth'], label=f'Smoothed {output_band} values', color='red')

plt.xlabel('Date')
plt.ylabel(f'{output_band} Value')
plt.title(f'{output_band} Time Series Analysis for Field')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Plot multi-year NDVI profiles by day of year
plt.figure(figsize=(12, 6))
for year, group in ndvi_data_df.groupby('year'):
    plt.plot(group['sort_day'], group[output_band], label=f'Year {year}', alpha=0.7)

plt.xlabel('Day of Year')
plt.ylabel(f'{output_band} Value')
plt.title(f'Multi-Year {output_band} Profiles')
plt.legend(title='Year')
plt.grid(True)
plt.show()


In [ ]:
# Scatter plot of NDVI values by year
plt.figure(figsize=(10, 6))
plt.scatter(ndvi_data_df['year'], ndvi_data_df[output_band], s=10, alpha=0.6, label=f'Individual {output_band} points')
plt.xlabel('Year')
plt.ylabel(f'{output_band} Value')
plt.title(f'Scatter Plot of {output_band} by Year')
plt.grid(True)
plt.legend()
plt.show()
